## Facade

---

> **In one line.** A facade is a small set $S$ of high-level operations, each realized as a *coordinated sequence* of many low-level subsystem calls. It shrinks the interface the client sees down to just $\lvert S\rvert$ entry points, while the system's full set of operations keeps running underneath.

### 1. The two layers

Let $C_1, C_2, \dots, C_n$ (e.g. TV, SoundSystem, StreamingService, Lights) represent $n$ subsystems whose combined product forms a large one big system $C$:
$$C = C_1 \times C_2 \times \cdots \times C_n,$$

Let $\Sigma$ represent the **state** of $C$ that is the combined state of all $n$ subsystems at once. Each subsystem exposes its own operations $\operatorname{Ops}(C_i)$; and we can write the full low-level **operation set** as the disjoint union

$$\operatorname{Ops}(C) \;:=\; \bigsqcup_{i=1}^{n} \operatorname{Ops}(C_i),$$

This set is **large**. Against it sits $S$, the **simplified interface**: a small set of high-level operations the client is allowed to call (e.g. just `watch_movie` and `end_movie`).

### 2. What a facade is

The facade assigns to each high-level operation $s \in S$ a finite **sequence** of subsystem operations that implement it:

$$\boxed{\,F : S \longrightarrow \operatorname{Ops}(C)^{*}\,}, \qquad F(s) = (g_1, g_2, \dots, g_k), \quad g_j \in \operatorname{Ops}(C).$$

Here $\operatorname{Ops}(C)^{*}$ is the set of finite words (sequences) over the low-level operations defined in §1. Executing $s$ runs that sequence in order on the system state $\Sigma$:

$$[\![\, s \,]\!] \;=\; [\![\, g_k \,]\!] \circ \cdots \circ [\![\, g_1 \,]\!] \;:\; \Sigma \to \Sigma.$$

**Direction matters.** The facade *exposes* $S$ and *dispatches* into $C$. Data flows

$$\text{client} \xrightarrow{\;s \in S\;} F \xrightarrow{\;F(s)\;} \operatorname{Ops}(C).$$

So $F$ goes **from the simple interface to the complex one**, expanding one call into many. (This is why $F : C \to S$ would be backwards: nothing collapses the whole system into a single op.)

### 3. Construction vs. dispatch (the OOP split)

Two maps live here; keep them apart.

- **Construction.** The facade holds references to the subsystem instances. This is a map $C_1 \times \cdots \times C_n \to \mathrm{Facade}$, matching `__init__(self, tv, sound, ...)`.
- **Dispatch.** Each method $s$ runs the composite $F(s)$ from §2, matching `watch_movie(self)` etc.

### 4. The defining inequality

$$\boxed{\,\lvert S\rvert \;\ll\; \lvert\operatorname{Ops}(C)\rvert\,}$$

Few entry points, much hidden machinery. A single call in $S$ may trigger many across $\operatorname{Ops}(C)$.

### 5. The three conditions

| Condition | Statement |
|---|---|
| **Simplification** | $\lvert S\rvert \ll \lvert\operatorname{Ops}(C)\rvert$: the facade exposes far fewer operations than the system contains. One $s \in S$ may fan out into many subsystem calls. |
| **Hidden, not removed** | the subsystems still exist and still run; $F$ only *re-packages* their operations. The complexity is concealed, not deleted. |
| **Subsystem independence** | each $C_i$ remains directly usable on its own. The facade is an **optional** convenience layer, not a gatekeeper. |

&nbsp;

> *Aside on “projection.”* It is tempting to call this a projection onto a subspace, but it is not a linear projection. More precisely, the facade picks a small set of **macro-operations**, each a composite *word* $F(s) \in \operatorname{Ops}(C)^{*}$ in the subsystem operations. The reduction is in the **number of entry points** ($\lvert S\rvert \ll \lvert\operatorname{Ops}(C)\rvert$), not in the state space.

> 🎬 Pressing “Play” is one action in $S$. Behind it: authenticate account, check subscription, load the video server, select resolution, buffer the stream, start DRM — all of that is $\operatorname{Ops}(C)$. You see one button; the complexity still runs.

### Exercise 1 — Home Theatre Facade

---

**Scenario:** The subsystems `TV`, `SoundSystem`, `StreamingService`, `Lights` each have their own methods (this is $\operatorname{Ops}(C)$). One `watch_movie()` call must orchestrate all of them (this is one $s \in S$).

**Your task:** Build `HomeTheatreFacade` so that $|S| = 2$ methods (`watch_movie`, `end_movie`) coordinate the many subsystem calls $|\operatorname{Ops}(C)|$.

```python
facade = HomeTheatreFacade(TV(), SoundSystem(), StreamingService(), Lights())
facade.watch_movie("Inception")   # one call in S -> many calls in Ops(C)
facade.end_movie()                 # the reverse sequence
```

**Hints**

- The facade holds references to all subsystems: `self.tv = tv`, etc. These are $C_1, C_2, \dots, C_n$ stored inside $F$.
- `watch_movie()` sequences the subsystem calls in the right order. That ordered sequence *is* $F(s) = (g_1, \dots, g_k)$: one $s \in S$ triggering the full $\operatorname{Ops}(C)$.
- `end_movie()` is the same idea in reverse — tear the session down in the opposite order.

In [ ]:
# --------------------------------
# Subsystems (Ops(C)) — each is a complex component; you do not change these

class TV:
    def on(self):            print("TV: on")
    def off(self):           print("TV: off")
    def set_input(self, src): print(f"TV: input -> {src}")

class SoundSystem:
    def on(self):             print("Sound: on")
    def off(self):            print("Sound: off")
    def set_volume(self, v):  print(f"Sound: volume -> {v}")

class StreamingService:
    def connect(self):        print("Streaming: connected")
    def play(self, title):    print(f"Streaming: playing '{title}'")
    def stop(self):           print("Streaming: stopped")

class Lights:
    def dim(self, pct):       print(f"Lights: dimmed to {pct}%")
    def on(self):             print("Lights: on")

# --------------------------------
# Facade (F) — your task: expose just watch_movie / end_movie (S)
# and orchestrate the subsystem calls (Ops(C)) inside them

class HomeTheatreFacade:
    def __init__(self, tv, sound, streaming, lights):
        self.tv = tv                 # C_1 ... C_n stored inside F
        self.sound = sound
        self.streaming = streaming
        self.lights = lights

    def watch_movie(self, title):    # one s in S -> the sequence F(s)
        ...                          # dim lights; tv on + set_input; sound on + set_volume;
                                     # streaming connect + play(title)

    def end_movie(self):             # the reverse sequence
        ...                          # streaming stop; sound off; tv off; lights on

# --------------------------------
facade = HomeTheatreFacade(TV(), SoundSystem(), StreamingService(), Lights())
facade.watch_movie("Inception")
print("--------")
facade.end_movie()

### Exercise 2 — Order Processing Facade

---

**Scenario:** The backend systems `InventorySystem`, `PaymentGateway`, `ShippingService`, `EmailNotifier` make up $\operatorname{Ops}(C)$. The client needs just one method: `place_order()` (this is $S$, with $|S| = 1$).

**Your task:** Write `OrderFacade.place_order(item, quantity, payment_info)` coordinating all four subsystems. Handle a payment failure gracefully (do not ship; notify the customer).

```python
facade = OrderFacade(InventorySystem(), PaymentGateway(), ShippingService(), EmailNotifier())
facade.place_order("Keyboard", 2, payment_info="valid-card")   # full happy path
facade.place_order("Mouse", 1, payment_info="bad-card")        # payment fails -> abort
```

**Hints**

- The client calling `place_order()` never imports `InventorySystem` — it only sees $S$. That is the simplification: $|S| = 1$ versus $|\operatorname{Ops}(C)| = $ many across four subsystems.
- Sequence the calls: check stock → charge payment → (only on success) reserve stock, schedule shipping, send a confirmation email. On payment failure, stop early and send a failure notice. The whole ordered sequence is $F(s)$.

In [ ]:
# --------------------------------
# Subsystems (Ops(C)) — you do not change these

class InventorySystem:
    def check_stock(self, item, qty):
        print(f"Inventory: checking {qty} x {item}")
        return True                           # pretend always in stock
    def reserve(self, item, qty):
        print(f"Inventory: reserved {qty} x {item}")

class PaymentGateway:
    def charge(self, payment_info, amount):
        ok = payment_info != "bad-card"       # 'bad-card' simulates a decline
        print(f"Payment: charging {amount} with {payment_info} -> {'OK' if ok else 'DECLINED'}")
        return ok

class ShippingService:
    def schedule(self, item, qty):
        print(f"Shipping: scheduled {qty} x {item}")

class EmailNotifier:
    def send(self, message):
        print(f"Email: {message}")

# --------------------------------
# Facade (F) — your task: expose just place_order (S), orchestrate the four subsystems

class OrderFacade:
    def __init__(self, inventory, payment, shipping, email):
        self.inventory = inventory            # C_1 ... C_n stored inside F
        self.payment = payment
        self.shipping = shipping
        self.email = email

    def place_order(self, item, quantity, payment_info):   # the single s in S
        # 1) check stock
        # 2) charge payment; if it fails -> notify and STOP (graceful)
        # 3) on success: reserve stock, schedule shipping, send confirmation
        ...

# --------------------------------
facade = OrderFacade(InventorySystem(), PaymentGateway(), ShippingService(), EmailNotifier())
facade.place_order("Keyboard", 2, payment_info="valid-card")
print("--------")
facade.place_order("Mouse", 1, payment_info="bad-card")